In [1]:
!pip install -U transformers faiss-gpu-cu11 ultralytics accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 37.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 21.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.9 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggi

# Pre-Processing

## Mapping the Metadata from JSON to Hashmap for Faster Lookups

In [2]:
import json

def load_metadata(json_path):
    """
    Loads the description JSON and converts it into a dictionary 
    keyed by item_id for O(1) lookups.
    """
    with open(json_path, 'r', encoding="utf-8") as f:
        data = json.load(f)
        
    metadata_dict = {}
    for entry in data:
        item_id = entry['item']
        metadata_dict[item_id] = {
            'color': entry.get('color', 'Unknown'),
            'description': " ".join(entry.get('description', []))
        }
        
    print(f"Loaded metadata for {len(metadata_dict)} unique items.")
    return metadata_dict

# Example usage:
# meta = load_metadata('list_description_inshop.json')
# print(meta['id_00000001'])

## Parsing Bounding Boxes

In [3]:
import pandas as pd

def load_bounding_boxes(bbox_filepath):
    """
    Reads the bounding box text file and returns a dictionary
    mapping the image path to its bounding box coordinates.
    """
    # Skip the first row (total count)
    df = pd.read_csv(bbox_filepath, sep=r'\s+', skiprows=1)
    df.columns = df.columns.str.strip()
    
    bbox_dict = {}
    for _, row in df.iterrows():
        img_name = row['image_name']
        # Extract coordinates: [x_min, y_min, x_max, y_max]
        coords = [row['x_1'], row['y_1'], row['x_2'], row['y_2']]
        bbox_dict[img_name] = coords
        
    print(f"Loaded bounding boxes for {len(bbox_dict)} images.")
    return bbox_dict

# Example usage:
# bboxes = load_bounding_boxes('list_bbox_inshop.txt')
# print(bboxes['img/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg'])

## Create the Dataset Class

In [4]:
import os
from PIL import Image
from torch.utils.data import Dataset

class DeepFashionDataset(Dataset):
    def __init__(self, partition_file, img_base_dir, split, transform=None, bbox_dict=None):
        """
        Args:
            partition_file (str): Path to list_eval_partition.txt
            img_base_dir (str): Path to the folder containing the 'img' directory
            split (str): One of 'train', 'gallery', or 'query'
            transform (callable, optional): PyTorch transforms to apply to the image
            bbox_dict (dict, optional): Dictionary containing bounding boxes
        """
        self.img_base_dir = img_base_dir
        self.transform = transform
        self.bbox_dict = bbox_dict
        
        # Load the partition file
        df = pd.read_csv(partition_file, sep=r'\s+', skiprows=1)
        df.columns = df.columns.str.strip()
        
        # Filter for the requested split
        self.data = df[df['evaluation_status'] == split].reset_index(drop=True)
        print(f"Initialized {split} dataset with {len(self.data)} images.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        rel_path = row['image_name']
        item_id = row['item_id']
        
        # Construct full image path and load it
        img_path = os.path.join(self.img_base_dir, rel_path)
        
        # Convert to RGB to ensure consistency (some images might be grayscale/RGBA)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # Get bounding box if available
        bbox = []
        if self.bbox_dict and rel_path in self.bbox_dict:
            bbox = self.bbox_dict[rel_path]
            
        return {
            'image': image,
            'item_id': item_id,
            'rel_path': rel_path,
            'bbox': bbox
        }

## Test Phase-1

In [6]:
from torchvision import transforms

# 1. Load the helper dictionaries
meta_dict = load_metadata('/kaggle/input/datasets/dveers/vr-final-ds/list_description_inshop.json')
bbox_dict = load_bounding_boxes('/kaggle/input/datasets/dveers/vr-final-ds/list_bbox_inshop.txt')

# 2. Define a simple transform (Convert to Tensor and resize)
# You will change this later depending on what YOLO or CLIP needs
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 3. Initialize the Gallery dataset
gallery_dataset = DeepFashionDataset(
    partition_file='/kaggle/input/datasets/dveers/vr-final-ds/list_eval_partition.txt',
    img_base_dir='/kaggle/input/datasets/dveers/vr-final-ds/img', # Change if your 'img' folder is somewhere else
    split='gallery',
    transform=basic_transform,
    bbox_dict=bbox_dict
)

# 4. Fetch the first item to test
sample = gallery_dataset[0]
sample_item_id = sample['item_id']

print("\n--- TEST FETCH ---")
print(f"Image Shape: {sample['image'].shape}")
print(f"Item ID: {sample_item_id}")
print(f"Bounding Box: {sample['bbox']}")
print(f"Metadata: {meta_dict.get(sample_item_id, 'No metadata found')}")

Loaded metadata for 7982 unique items.
Loaded bounding boxes for 52712 images.
Initialized gallery dataset with 12612 images.

--- TEST FETCH ---
Image Shape: torch.Size([3, 224, 224])
Item ID: id_00000001
Bounding Box: [50, 49, 208, 235]
Metadata: {'color': 'Cream', 'description': 'This sheer Georgette top features a high collar and shirred shoulders. Complete with long sleeves and buttoned cuffs.  Unlined Lightweight, woven 100% polyester 24" full length, 38" chest, 38" waist, 24" sleeve length Measured from Small Hand wash cold Imported'}


# Step-1: Product Localization and Offline Indexing

## Convert BBoxes to YOLO Format

In [7]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm

# --- CONFIGURATION ---
PARTITION_FILE = "/kaggle/input/datasets/dveers/vr-final-ds/list_eval_partition.txt"
BBOX_FILE = "/kaggle/input/datasets/dveers/vr-final-ds/list_bbox_inshop.txt"
BASE_DIR = os.path.abspath("/kaggle/input/datasets/dveers/vr-final-ds") 

# --- NEW: Writable directories for YOLO ---
YOLO_IMG_DIR = "/kaggle/working/yolo_data/images"
YOLO_LBL_DIR = "/kaggle/working/yolo_data/labels"

os.makedirs(YOLO_IMG_DIR, exist_ok=True)
os.makedirs(YOLO_LBL_DIR, exist_ok=True)

def convert_to_yolo_format(img_width, img_height, x1, y1, x2, y2):
    dw = 1.0 / img_width
    dh = 1.0 / img_height
    x_center = (x1 + x2) / 2.0
    y_center = (y1 + y2) / 2.0
    return x_center * dw, y_center * dh, (x2 - x1) * dw, (y2 - y1) * dh

def setup_yolo_data():
    print("Loading datasets...")
    df_part = pd.read_csv(PARTITION_FILE, sep=r'\s+', skiprows=1)
    df_bbox = pd.read_csv(BBOX_FILE, sep=r'\s+', skiprows=1)
    
    df_part.columns = df_part.columns.str.strip()
    df_bbox.columns = df_bbox.columns.str.strip()

    bbox_dict = {row['image_name']: [row['x_1'], row['y_1'], row['x_2'], row['y_2']] 
                 for _, row in df_bbox.iterrows()}

    train_paths = []
    query_paths = []
    
    print("Generating labels and symlinks in /kaggle/working/...")
    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        rel_path = row['image_name']
        status = row['evaluation_status']
        
        if status not in ['train', 'query']:
            continue
            
        abs_input_img_path = os.path.join(BASE_DIR, "img", rel_path)
        
        if os.path.exists(abs_input_img_path) and rel_path in bbox_dict:
            
            # Create a safe, flat filename (e.g., "img_WOMEN_Dresses_id_..._front.jpg")
            safe_name = rel_path.replace("/", "_")
            txt_name = os.path.splitext(safe_name)[0] + ".txt"
            
            working_img_path = os.path.join(YOLO_IMG_DIR, safe_name)
            working_lbl_path = os.path.join(YOLO_LBL_DIR, txt_name)
            
            # 1. Create a symlink to the read-only image (Takes 0 bytes of disk space)
            if not os.path.exists(working_img_path):
                os.symlink(abs_input_img_path, working_img_path)
                
            # Add the new writable path to the tracking lists
            if status == 'train':
                train_paths.append(working_img_path)
            elif status == 'query':
                query_paths.append(working_img_path)
                
            # 2. Generate and write the YOLO label file to the writable labels folder
            x1, y1, x2, y2 = bbox_dict[rel_path]
            try:
                with Image.open(abs_input_img_path) as img:
                    w, h = img.size
                
                yolo_coords = convert_to_yolo_format(w, h, x1, y1, x2, y2)
                
                with open(working_lbl_path, 'w') as f:
                    f.write(f"0 {yolo_coords[0]:.6f} {yolo_coords[1]:.6f} {yolo_coords[2]:.6f} {yolo_coords[3]:.6f}\n")
            except Exception as e:
                print(f"Failed on {abs_input_img_path}: {e}")

    # 3. Write the lists to .txt files in the working directory
    with open('/kaggle/working/yolo_train_paths.txt', 'w') as f:
        f.write('\n'.join(train_paths))
    with open('/kaggle/working/yolo_query_paths.txt', 'w') as f:
        f.write('\n'.join(query_paths))
        
    print(f"\nDone! Tracked {len(train_paths)} training images and {len(query_paths)} query images.")

if __name__ == "__main__":
    setup_yolo_data()

Loading datasets...
Generating labels and symlinks in /kaggle/working/...


100%|██████████| 52712/52712 [03:30<00:00, 250.67it/s]


Done! Tracked 25882 training images and 14218 query images.


## Create YAML for YOLO

In [8]:
import os

# Set the root path to your Kaggle working directory where the .txt files are saved
WORKING_DIR = "/kaggle/working"

yaml_content = f"""# deepfashion.yaml
path: {WORKING_DIR}
train: yolo_train_paths.txt
val: yolo_query_paths.txt

names:
  0: clothing
"""

with open('/kaggle/working/deepfashion.yaml', 'w') as f:
    f.write(yaml_content)

print("Successfully created corrected deepfashion.yaml")

Successfully created corrected deepfashion.yaml


## Training Script for YOLO

In [10]:
from ultralytics import YOLO

# Load the YOLO11 Small model
model = YOLO('yolo11s.pt') 

print("Starting YOLO11 Small fine-tuning...")
results = model.train(
    data='/kaggle/working/deepfashion.yaml', 
    epochs=15,       
    imgsz=640,              
    batch=16,        
    device=0,        
    plots=True,
    workers=4        
)

print("Training complete! Best weights saved to: runs/detect/train/weights/best.pt")

Starting YOLO11 Small fine-tuning...
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/deepfashion.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=

## Cropping Function for YOLO

In [5]:
import os
import random
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from torch.optim import AdamW
from ultralytics import YOLO
from tqdm.notebook import tqdm
from IPython.display import display

# --- KAGGLE DEVICE CONFIG ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- PATHS ---
PARTITION_FILE = "/kaggle/input/datasets/dveers/vr-final-ds/list_eval_partition.txt"
BBOX_FILE = "/kaggle/input/datasets/dveers/vr-final-ds/list_bbox_inshop.txt"
BASE_DIR = os.path.abspath("/kaggle/input/datasets/dveers/vr-final-ds")
YOLO_MODEL_PATH = '/kaggle/input/models/dveers/yolov11-vr-final/pytorch/default/3/runs/detect/train/weights/best.pt'

# --- LOAD YOLO ---
print("Loading fine-tuned YOLO...")
yolo_model = YOLO(YOLO_MODEL_PATH)

def get_cropped_image(img_path, bbox=None):
    """Crops the image using bounding boxes or YOLO."""
    img = Image.open(img_path).convert('RGB')
    
    if bbox is not None and len(bbox) == 4:
        return img.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
        
    # FIX: Added device targeting and half-precision for Kaggle GPU speed
    use_half = True if DEVICE == "cuda" else False
    results = yolo_model(img, conf=0.25, verbose=False, half=use_half)
    
    if len(results[0].boxes) > 0:
        boxes = results[0].boxes.data
        best_box = boxes[boxes[:, 4].argmax()] 
        x1, y1, x2, y2 = best_box[:4].tolist()
        return img.crop((x1, y1, x2, y2))
        
    return img

# --- Quick Test ---
# test_rel_path = 'img/WOMEN/Leggings/id_00000225/03_2_side.jpg'
# test_abs_path = os.path.join(BASE_DIR, test_rel_path)
# if os.path.exists(test_abs_path):
#     cropped = get_cropped_image(test_abs_path)
#     display(cropped) 
# else:
#     print(f"Test image not found at: {test_abs_path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading fine-tuned YOLO...


## Triplet Dataset (Anchor, Positive, Negative)

In [14]:
# Load bounding boxes into memory for fast training lookups
print("Loading bounding boxes...")
df_bbox = pd.read_csv(BBOX_FILE, sep=r'\s+', skiprows=1)
df_bbox.columns = df_bbox.columns.str.strip()
bbox_dict = {row['image_name']: [row['x_1'], row['y_1'], row['x_2'], row['y_2']] 
             for _, row in df_bbox.iterrows()}

class DeepFashionTripletDataset(Dataset):
    def __init__(self, partition_file, bbox_data, base_dir, processor):
        self.base_dir = base_dir
        self.processor = processor
        self.bbox_dict = bbox_data
        
        df = pd.read_csv(partition_file, sep=r'\s+', skiprows=1)
        df.columns = df.columns.str.strip()
        self.train_df = df[df['evaluation_status'] == 'train'].reset_index(drop=True)
        
        self.item_groups = self.train_df.groupby('item_id')['image_name'].apply(list).to_dict()
        self.item_ids = list(self.item_groups.keys())
        
        self.valid_items = [i for i in self.item_ids if len(self.item_groups[i]) >= 2]
        print(f"Dataset ready. {len(self.valid_items)} valid items for Triplet matching.")

    def __len__(self):
        return len(self.valid_items)

    def process_image(self, rel_path):
        abs_path = os.path.join(self.base_dir, "img", rel_path)
        bbox = self.bbox_dict.get(rel_path, None)
        cropped_img = get_cropped_image(abs_path, bbox)
        return self.processor(images=cropped_img, return_tensors="pt")['pixel_values'].squeeze(0)

    def __getitem__(self, idx):
        anchor_item = self.valid_items[idx]
        anchor_path, positive_path = random.sample(self.item_groups[anchor_item], 2)
        
        negative_item = random.choice(self.item_ids)
        while negative_item == anchor_item:
            negative_item = random.choice(self.item_ids)
            
        negative_path = random.choice(self.item_groups[negative_item])
        
        anchor_tensor = self.process_image(anchor_path)
        positive_tensor = self.process_image(positive_path)
        negative_tensor = self.process_image(negative_path)
        
        return anchor_tensor, positive_tensor, negative_tensor

Loading bounding boxes...


## CLIP Fine-Tuning Loop

In [17]:
import gc
import os
import random
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import CLIPProcessor, CLIPModel
from torch.optim import AdamW
from tqdm.notebook import tqdm 

# --- KAGGLE T4 OPTIMIZED CONFIG ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64       
EPOCHS = 5           
LEARNING_RATE = 1e-6 
SEEDS = [543, 45, 56]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print("Preparing Dataset...")
# Assuming DeepFashionTripletDataset is already defined in your notebook
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
dataset = DeepFashionTripletDataset(PARTITION_FILE, bbox_dict, BASE_DIR, processor)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

for current_seed in SEEDS:
    print(f"\n" + "="*50)
    print(f"STARTING FINE-TUNING FOR SEED: {current_seed}")
    print("="*50)
    
    set_seed(current_seed)
    
    # RELOAD A FRESH MODEL EVERY SEED SO WE DON'T OVERWRITE PREVIOUS TRAINING
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
    
    # Freeze Text, Train Vision
    for param in model.text_model.parameters(): param.requires_grad = False
    for param in model.vision_model.parameters(): param.requires_grad = True
    for param in model.visual_projection.parameters(): param.requires_grad = True

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    criterion = nn.TripletMarginLoss(margin=1.0, p=2)
    scaler = torch.amp.GradScaler('cuda')

    model.train()

    try:
        for epoch in range(EPOCHS):
            total_loss = 0
            pbar = tqdm(dataloader, desc=f"Seed {current_seed} | Epoch {epoch+1}/{EPOCHS}")
            
            for anchor, positive, negative in pbar:
                anchor, positive, negative = anchor.to(DEVICE), positive.to(DEVICE), negative.to(DEVICE)
                optimizer.zero_grad()
                
                with torch.amp.autocast('cuda'):
                    # Vision encoding
                    anchor_out = model.vision_model(pixel_values=anchor)
                    positive_out = model.vision_model(pixel_values=positive)
                    negative_out = model.vision_model(pixel_values=negative)
                    
                    # Projection
                    anchor_embed = model.visual_projection(anchor_out.pooler_output)
                    positive_embed = model.visual_projection(positive_out.pooler_output)
                    negative_embed = model.visual_projection(negative_out.pooler_output)
                    
                    # Normalization
                    anchor_embed = anchor_embed / anchor_embed.norm(p=2, dim=-1, keepdim=True)
                    positive_embed = positive_embed / positive_embed.norm(p=2, dim=-1, keepdim=True)
                    negative_embed = negative_embed / negative_embed.norm(p=2, dim=-1, keepdim=True)
                    
                    loss = criterion(anchor_embed, positive_embed, negative_embed)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
                pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
                
            print(f"Average Loss (Seed {current_seed}, Epoch {epoch+1}): {total_loss / len(dataloader):.4f}")
            
        # Save this specific seed's model
        save_path = f"/kaggle/working/finetuned_clip_full_{current_seed}"
        print(f"\nSaving model to {save_path}...")
        model.save_pretrained(save_path)
        processor.save_pretrained(save_path)
        
    except Exception as e:
        print(f"\n[!] Error during training seed {current_seed}: {e}")
    finally:
        # Clear GPU memory before starting the next seed
        del model, optimizer, scaler
        if 'anchor' in locals(): del anchor, positive, negative, anchor_embed, loss
        gc.collect()
        torch.cuda.empty_cache()

print("\n \t Finished Fine-tuning on all Seeds Successfully!")

Preparing Dataset...
Dataset ready. 3985 valid items for Triplet matching.

STARTING FINE-TUNING FOR SEED: 543


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Seed 543 | Epoch 1/5:   0%|          | 0/63 [00:00<?, ?it/s]

Average Loss (Seed 543, Epoch 1): 0.6932


...

Average Loss (Seed 56, Epoch 4): 0.2556


Seed 56 | Epoch 5/5:   0%|          | 0/63 [00:00<?, ?it/s]

Average Loss (Seed 56, Epoch 5): 0.2297

Saving model to /kaggle/working/finetuned_clip_full_56...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 	 Finished Fine-tuning on all Seeds Successfully!


# Index Builders

## Creating Gallery Metadata

In [3]:
import os
import json
import torch
import gc
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from ultralytics import YOLO
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# --- 1. DUAL-GPU CONFIGURATION ---
print("Checking GPU availability...")
if torch.cuda.device_count() >= 2:
    print(f"Dual GPUs detected! Splitting the load...")
    DEVICE_YOLO = "cuda:0"
    DEVICE_BLIP = "cuda:1"
    BLIP_DEVICE_MAP = {"": 1} # Forces BLIP onto GPU 1
else:
    print("Warning: Only 1 GPU detected. Running everything on cuda:0...")
    DEVICE_YOLO = "cuda:0"
    DEVICE_BLIP = "cuda:0"
    BLIP_DEVICE_MAP = {"": 0}

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
PARTITION_FILE = os.path.join(BASE_DIR, "list_eval_partition.txt")
YOLO_MODEL_PATH = "/kaggle/input/models/dveers/yolov11-vr-final/pytorch/default/3/runs/detect/train/weights/best.pt"
OUTPUT_META_PATH = "/kaggle/working/gallery_metadata.json"

# --- 2. LOAD PARTITION DATA ---
print("\n1. Loading Partition Data...")
df_part = pd.read_csv(PARTITION_FILE, sep=r'\s+', skiprows=1)
df_part.columns = df_part.columns.str.strip()

# We only process the gallery images for the offline index
gallery_df = df_part[df_part['evaluation_status'] == 'gallery'].copy()
gallery_df['image_name'] = gallery_df['image_name'].str.strip()
print(f"Found {len(gallery_df)} Gallery images to process.")

# --- 3. LOAD MODELS (SPLIT ACROSS GPUS) ---
print(f"\n2. Loading YOLO to {DEVICE_YOLO}...")
yolo_model = YOLO(YOLO_MODEL_PATH)

print(f"Loading massive BLIP-2 to {DEVICE_BLIP} (This takes a moment)...")
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", 
    torch_dtype=torch.float16, 
    device_map=BLIP_DEVICE_MAP
)

def get_cropped_image(img_path):
    try:
        img = Image.open(img_path).convert('RGB')
        # Explicitly tell YOLO to use half precision if on GPU
        use_half = True if torch.cuda.is_available() else False
        results = yolo_model(img, conf=0.25, verbose=False, half=use_half, device=0 if DEVICE_YOLO=="cuda:0" else "cpu")
        
        if len(results[0].boxes) > 0:
            boxes = results[0].boxes.data
            best_box = boxes[boxes[:, 4].argmax()] 
            x1, y1, x2, y2 = best_box[:4].tolist()
            return img.crop((x1, y1, x2, y2))
        return img
    except Exception as e:
        print(f"Error reading {img_path}: {e}")
        return None

# --- 4. SEMANTIC CAPTIONING LOOP ---
print("\n3. Starting Semantic Captioning (This will take a while!)...")
metadata_list = []
faiss_index_counter = 0

for idx, row in tqdm(gallery_df.iterrows(), total=len(gallery_df)):
    rel_path = row['image_name']
    item_id = row['item_id']
    
    abs_path = os.path.join(BASE_DIR, "img", rel_path) 
    
    if not os.path.exists(abs_path):
        continue
        
    cropped_img = get_cropped_image(abs_path) # Runs on GPU 0
    if cropped_img is None:
        continue
        
    # Generate the text caption using BLIP-2 on GPU 1
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        # Explicitly send image tensors to the GPU holding BLIP-2
        inputs = blip_processor(images=cropped_img, return_tensors="pt").to(DEVICE_BLIP, torch.float16)
        
        generated_ids = blip_model.generate(**inputs, max_new_tokens=20)
        generated_caption = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        
    # Build the dictionary for this item
    metadata_entry = {
        "faiss_id": faiss_index_counter,
        "item_id": item_id,
        "image_path": rel_path, 
        "generated_caption": generated_caption
    }
    
    metadata_list.append(metadata_entry)
    faiss_index_counter += 1

print("\n4. Saving Metadata to JSON...")
with open(OUTPUT_META_PATH, "w", encoding='utf-8') as f:
    json.dump(metadata_list, f, indent=4)
    
print(f"Success! Generated captions and saved metadata for {len(metadata_list)} items to {OUTPUT_META_PATH}")

# --- 5. AGGRESSIVE DUAL-GPU MEMORY CLEANUP ---
print("\nFlushing VRAM across all GPUs...")

if 'inputs' in locals(): del inputs
if 'generated_ids' in locals(): del generated_ids
if 'cropped_img' in locals(): del cropped_img
del yolo_model, blip_model, blip_processor

gc.collect()
torch.cuda.empty_cache()

if torch.cuda.device_count() >= 2:
    with torch.cuda.device(1):
        torch.cuda.empty_cache()
    with torch.cuda.device(0):
        torch.cuda.empty_cache()

print("GPU Memory wiped.")

Checking GPU availability...

1. Loading Partition Data...
Found 12612 Gallery images to process.

2. Loading YOLO to cuda:0...
Loading massive BLIP-2 to cuda:0 (This takes a moment)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]


3. Starting Semantic Captioning (This will take a while!)...


  0%|          | 0/12612 [00:00<?, ?it/s]


4. Saving Metadata to JSON...
Success! Generated captions and saved metadata for 12612 items to /kaggle/working/gallery_metadata.json

Flushing VRAM across all GPUs...
GPU Memory wiped.


## Part A

In [4]:
import os, json, torch, faiss, gc
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
YOLO_MODEL_PATH = "/kaggle/input/models/dveers/yolov11-vr-final/pytorch/default/3/runs/detect/train/weights/best.pt"

# Use Pre-Trained CLIP
CLIP_PATH = "openai/clip-vit-base-patch32"
INDEX_OUTPUT_PATH = "/kaggle/working/gallery_index_pretrained.faiss"
EXISTING_META_PATH = "/kaggle/input/datasets/dveers/vr-final-ds/gallery_metadata.json"

print("Loading Models for Part A Indexing...")
yolo_model = YOLO(YOLO_MODEL_PATH)
clip_processor = CLIPProcessor.from_pretrained(CLIP_PATH)
clip_model = CLIPModel.from_pretrained(CLIP_PATH).to(DEVICE)

def get_cropped_image(img_path):
    img = Image.open(img_path).convert('RGB')
    results = yolo_model(img, conf=0.25, verbose=False, half=(DEVICE=="cuda"))
    if len(results[0].boxes) > 0:
        return img.crop(results[0].boxes.data[results[0].boxes.data[:, 4].argmax()][:4].tolist())
    return img

with open(EXISTING_META_PATH, "r") as f:
    metadata = json.load(f)

embeddings = []
print("Building Vision-Only Index for Pre-trained CLIP...")

for item in tqdm(metadata):
    abs_path = os.path.join(BASE_DIR, "img", item['image_path'])
    if not os.path.exists(abs_path): continue
        
    cropped_img = get_cropped_image(abs_path)
    
    with torch.no_grad(), torch.amp.autocast(device_type=DEVICE, dtype=torch.float16):
        # Vision Only (Alpha = 1)
        clip_inputs = clip_processor(images=cropped_img, return_tensors="pt").to(DEVICE)
        vision_out = clip_model.vision_model(pixel_values=clip_inputs.pixel_values)
        img_embed = clip_model.visual_projection(vision_out.pooler_output)
        
        # Normalize
        img_embed = img_embed / img_embed.norm(p=2, dim=-1, keepdim=True)
        embeddings.append(img_embed.to(torch.float32).cpu().numpy()[0])

# The '32' is the 'M' parameter (number of connections per node), standard for HNSW
index = faiss.IndexHNSWFlat(512, 32)
index.add(np.array(embeddings, dtype=np.float32))
faiss.write_index(index, INDEX_OUTPUT_PATH)
print(f"Saved Part A Index to {INDEX_OUTPUT_PATH}")

Loading Models for Part A Indexing...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building Vision-Only Index for Pre-trained CLIP...


  0%|          | 0/12612 [00:00<?, ?it/s]

Saved Part A Index to /kaggle/working/gallery_index_pretrained.faiss


## Part B

In [5]:
import os, json, torch, faiss, gc
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from transformers import CLIPProcessor, CLIPModel

ALPHAS = [0.5, 0.8]
INDEX_PATH_B_05 = "/kaggle/working/gallery_index_pretrained_alpha05.faiss"
INDEX_PATH_B_08 = "/kaggle/working/gallery_index_pretrained_alpha08.faiss"

print("Building Cross-Modal Fused Indices for Part B (Alphas: 0.5 & 0.8)...")
embeddings_05 = []
embeddings_08 = []

for item in tqdm(metadata):
    abs_path = os.path.join(BASE_DIR, "img", item['image_path'])
    if not os.path.exists(abs_path): continue
        
    cropped_img = get_cropped_image(abs_path)
    caption = item['generated_caption']
    
    with torch.no_grad(), torch.amp.autocast(device_type=DEVICE_YOLO, dtype=torch.float16):
        # 1. Image Embedding
        img_inputs = clip_processor(images=cropped_img, return_tensors="pt").to(DEVICE_YOLO)
        vision_out = clip_model.vision_model(pixel_values=img_inputs.pixel_values)
        img_embed = clip_model.visual_projection(vision_out.pooler_output)
        img_embed = img_embed / img_embed.norm(p=2, dim=-1, keepdim=True)
        
        # 2. Text Embedding
        text_inputs = clip_processor(text=caption, return_tensors="pt", padding=True, truncation=True).to(DEVICE_YOLO)
        text_out = clip_model.text_model(**text_inputs)
        txt_embed = clip_model.text_projection(text_out.pooler_output)
        txt_embed = txt_embed / txt_embed.norm(p=2, dim=-1, keepdim=True)
        
        # 3. Double Fusion Logic (Saves 100% compute time!)
        fused_05 = (0.5 * img_embed) + (0.5 * txt_embed)
        fused_05 = fused_05 / fused_05.norm(p=2, dim=-1, keepdim=True)
        embeddings_05.append(fused_05.to(torch.float32).cpu().numpy()[0])
        
        fused_08 = (0.8 * img_embed) + (0.2 * txt_embed)
        fused_08 = fused_08 / fused_08.norm(p=2, dim=-1, keepdim=True)
        embeddings_08.append(fused_08.to(torch.float32).cpu().numpy()[0])

# Save both HNSW Indices
index_b_05 = faiss.IndexHNSWFlat(512, 32)
index_b_05.add(np.array(embeddings_05, dtype=np.float32))
faiss.write_index(index_b_05, INDEX_PATH_B_05)

index_b_08 = faiss.IndexHNSWFlat(512, 32)
index_b_08.add(np.array(embeddings_08, dtype=np.float32))
faiss.write_index(index_b_08, INDEX_PATH_B_08)

print(f"Saved Part B Indices to {INDEX_PATH_B_05} and {INDEX_PATH_B_08}")

Building Cross-Modal Fused Indices for Part B (Alphas: 0.5 & 0.8)...


  0%|          | 0/12612 [00:00<?, ?it/s]

Saved Part B Indices to /kaggle/working/gallery_index_pretrained_alpha05.faiss and /kaggle/working/gallery_index_pretrained_alpha08.faiss


## Part C

In [7]:
SEEDS = [543, 45, 56]
ALPHAS = [0.5, 0.8]

print("Building Part C Indices (3 Seeds x 2 Alphas = 6 Indices total)...")

for seed in SEEDS:
    print(f"\nBuilding Indices for Seed {seed}...")
    model_path = f"/kaggle/input/models/dveers/finetuned-clip-vr-final/pytorch/default/2/finetuned_clip_full_{seed}"
    
    ft_clip_processor = CLIPProcessor.from_pretrained(model_path)
    ft_clip_model = CLIPModel.from_pretrained(model_path).to(DEVICE_YOLO)
    
    embeddings_05, embeddings_08 = [], []
    
    for item in tqdm(metadata, desc=f"Encoding Seed {seed}"):
        abs_path = os.path.join(BASE_DIR, "img", item['image_path'])
        if not os.path.exists(abs_path): continue
            
        cropped_img = get_cropped_image(abs_path)
        caption = item['generated_caption']
        
        with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            img_inputs = ft_clip_processor(images=cropped_img, return_tensors="pt").to(DEVICE_YOLO)
            vision_out = ft_clip_model.vision_model(pixel_values=img_inputs.pixel_values)
            img_embed = ft_clip_model.visual_projection(vision_out.pooler_output)
            img_embed = img_embed / img_embed.norm(p=2, dim=-1, keepdim=True)
            
            text_inputs = ft_clip_processor(text=caption, return_tensors="pt", padding=True, truncation=True).to(DEVICE_YOLO)
            text_out = ft_clip_model.text_model(**text_inputs)
            txt_embed = ft_clip_model.text_projection(text_out.pooler_output)
            txt_embed = txt_embed / txt_embed.norm(p=2, dim=-1, keepdim=True)
            
            fused_05 = (0.5 * img_embed) + (0.5 * txt_embed)
            fused_05 = fused_05 / fused_05.norm(p=2, dim=-1, keepdim=True)
            embeddings_05.append(fused_05.to(torch.float32).cpu().numpy()[0])
            
            fused_08 = (0.8 * img_embed) + (0.2 * txt_embed)
            fused_08 = fused_08 / fused_08.norm(p=2, dim=-1, keepdim=True)
            embeddings_08.append(fused_08.to(torch.float32).cpu().numpy()[0])

    # Save HNSW Index for Alpha 0.5
    idx_05 = faiss.IndexHNSWFlat(512, 32)
    idx_05.add(np.array(embeddings_05, dtype=np.float32))
    faiss.write_index(idx_05, f"/kaggle/working/gallery_index_finetuned_{seed}_alpha05.faiss")
    
    # Save HNSW Index for Alpha 0.8
    idx_08 = faiss.IndexHNSWFlat(512, 32)
    idx_08.add(np.array(embeddings_08, dtype=np.float32))
    faiss.write_index(idx_08, f"/kaggle/working/gallery_index_finetuned_{seed}_alpha08.faiss")
    
    del ft_clip_model, ft_clip_processor, idx_05, idx_08
    gc.collect()
    torch.cuda.empty_cache()

Building Part C Indices (3 Seeds x 2 Alphas = 6 Indices total)...

Building Indices for Seed 543...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding Seed 543:   0%|          | 0/12612 [00:00<?, ?it/s]


Building Indices for Seed 45...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding Seed 45:   0%|          | 0/12612 [00:00<?, ?it/s]


Building Indices for Seed 56...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding Seed 56:   0%|          | 0/12612 [00:00<?, ?it/s]

# Ablation Study

In [6]:
import pandas as pd
import numpy as np
import torch
import os
import faiss
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# 1. Point to your partition file

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
PARTITION_FILE = os.path.join(BASE_DIR, "list_eval_partition.txt")

print("Loading query partition...")

# 2. Load the text file into a pandas DataFrame
df_part = pd.read_csv(PARTITION_FILE, sep=r'\s+', skiprows=1)

# 3. Clean the column names (scrubs hidden Windows characters)
df_part.columns = df_part.columns.str.strip()

# 4. Filter to create query_df (grabbing ONLY the 'query' status images)
query_df = df_part[df_part['evaluation_status'] == 'query'].copy()

# 5. Clean the image paths just to be safe
query_df['image_name'] = query_df['image_name'].str.strip()

print(f"Success! Loaded {len(query_df)} Query images ready for testing.")

def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

Loading query partition...
Success! Loaded 14218 Query images ready for testing.


In [13]:
import os
import zipfile
from IPython.display import display, FileLink

# --- Configuration ---
WORKING_DIR = "/kaggle/working"
ZIP_NAME = "all_faissa_indices.zip"
ZIP_PATH = os.path.join(WORKING_DIR, ZIP_NAME)

print("Scanning /kaggle/working/ for FAISS indices...")

# Find all files starting with 'gallery_index_' and ending with '.faiss'
faiss_files = [f for f in os.listdir(WORKING_DIR) if f.startswith("gallery_index_") and f.endswith(".faiss")]

if not faiss_files:
    print("❌ No FAISS indices found! Make sure your index builders have finished running.")
else:
    print(f"Found {len(faiss_files)} files. Compressing now...")
    
    # 1. Zip the files together using standard deflation
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in faiss_files:
            print(f" 📦 Adding: {file}")
            # arcname prevents it from zipping the entire folder tree, keeping it flat
            zipf.write(os.path.join(WORKING_DIR, file), arcname=file)
            
    # 2. Calculate the final archive size
    file_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    print(f"\n✅ Zipping complete! Final archive size: {file_size_mb:.2f} MB")
    
    # 3. Generate the stable download link
    print("\n⬇️ Click the link below to download your archive directly:")
    display(FileLink(ZIP_NAME))

Scanning /kaggle/working/ for FAISS indices...
Found 9 files. Compressing now...
 📦 Adding: gallery_index_finetuned_45_alpha08.faiss
 📦 Adding: gallery_index_pretrained_alpha08.faiss
 📦 Adding: gallery_index_finetuned_56_alpha08.faiss
 📦 Adding: gallery_index_pretrained.faiss
 📦 Adding: gallery_index_finetuned_543_alpha05.faiss
 📦 Adding: gallery_index_finetuned_543_alpha08.faiss
 📦 Adding: gallery_index_pretrained_alpha05.faiss
 📦 Adding: gallery_index_finetuned_56_alpha05.faiss
 📦 Adding: gallery_index_finetuned_45_alpha05.faiss

✅ Zipping complete! Final archive size: 215.80 MB

⬇️ Click the link below to download your archive directly:


/kaggle/working/all_faiss_indices.zip

## Part A

In [15]:
import os
import json
import torch
import faiss
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from transformers import CLIPProcessor, CLIPModel
from IPython.display import display, HTML

# --- 1. CONFIGURATION ---
print("Checking GPU availability...")
if torch.cuda.device_count() >= 2:
    DEVICE_YOLO = "cuda:0"
else:
    DEVICE_YOLO = "cuda:0" if torch.cuda.is_available() else "cpu"

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
INDEX_PATH_A = "/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices/gallery_index_pretrained.faiss" 
META_PATH = "/kaggle/input/datasets/dveers/vr-final-ds/gallery_metadata.json"
k_values = [5, 10, 15]

# --- 2. LOAD METADATA ---
with open(META_PATH, "r", encoding='utf-8') as f:
    metadata = json.load(f)

# --- 3. METRICS FUNCTION ---
def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

# --- 4. LOAD MODELS ---
print("Loading Pre-Trained CLIP for Part A...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE_YOLO)

index = faiss.read_index(INDEX_PATH_A)
total_metrics = {'Recall': {5: 0, 10: 0, 15: 0}, 'NDCG': {5: 0, 10: 0, 15: 0}, 'mAP': {5: 0, 10: 0, 15: 0}}
valid_queries = 0

# --- 5. EVALUATION LOOP ---
print("\n--- Running Evaluation for Part A (Vision-Only) ---")
for idx, row in tqdm(query_df.iterrows(), total=len(query_df)):
    abs_path = os.path.join(BASE_DIR, "img", row['image_name'])
    if not os.path.exists(abs_path): continue
    valid_queries += 1
    
    cropped_query = get_cropped_image(abs_path) 
    
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        # Vision Only (Alpha = 1)
        clip_inputs = clip_processor(images=cropped_query, return_tensors="pt").to(DEVICE_YOLO)
        vision_out = clip_model.vision_model(pixel_values=clip_inputs.pixel_values)
        query_embed = clip_model.visual_projection(vision_out.pooler_output)
        query_embed = query_embed / query_embed.norm(p=2, dim=-1, keepdim=True)
        
        # Search the Pretrained Vision-Only Index
        distances, indices = index.search(query_embed.to(torch.float32).cpu().numpy(), 15)
        candidates = [metadata[i] for i in indices[0]]
        
    retrieved_item_ids = [c['item_id'] for c in candidates]
    metrics = calculate_metrics(retrieved_item_ids, row['item_id'], k_values)
    
    for m in ['Recall', 'NDCG', 'mAP']:
        for k in k_values: total_metrics[m][k] += metrics[m][k]

# --- 6. OUTPUT RESULTS ---
print("\n" + "="*50 + "\nCONDITION A: Vision-Only Baseline (Alpha = 1)\n" + "="*50)
results_df = pd.DataFrame(total_metrics) / valid_queries
results_df.index = [f"@{k}" for k in k_values]
display(HTML(results_df.to_html(classes='table table-striped table-bordered text-center', float_format="%.4f")))

Checking GPU availability...
Loading Pre-Trained CLIP for Part A...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Evaluation for Part A (Vision-Only) ---


  0%|          | 0/14218 [00:00<?, ?it/s]


CONDITION A: Vision-Only Baseline (Alpha = 1)


,Recall,NDCG,mAP
@5,0.4240,0.4444,0.3193
@10,0.4881,0.5048,0.3139
@15,0.5224,0.5364,0.3069


## Part B

In [7]:
import os
import json
import torch
import faiss
import gc
import pandas as pd
import numpy as np
import torch.nn.functional as F
from tqdm.notebook import tqdm
from transformers import CLIPProcessor, CLIPModel, Blip2Processor, Blip2ForConditionalGeneration
from IPython.display import display, HTML

# --- 1. CONFIGURATION & DEVICES ---
print("Checking GPU availability...")
if torch.cuda.device_count() >= 2:
    DEVICE_YOLO = "cuda:0"
    DEVICE_BLIP = "cuda:1"
    BLIP_DEVICE_MAP = {"": 1}
else:
    DEVICE_YOLO = "cuda:0" if torch.cuda.is_available() else "cpu"
    DEVICE_BLIP = DEVICE_YOLO
    BLIP_DEVICE_MAP = {"": 0}

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
META_PATH = "/kaggle/input/datasets/dveers/vr-final-ds/gallery_metadata.json"
INDEX_PATH_B_05 = "/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices/gallery_index_pretrained_alpha05.faiss"
INDEX_PATH_B_08 = "/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices/gallery_index_pretrained_alpha08.faiss"
ALPHAS = [0.5, 0.8]
k_values = [5, 10, 15]

# --- 2. LOAD METADATA & METRICS ---
with open(META_PATH, "r", encoding='utf-8') as f:
    metadata = json.load(f)

def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

# --- 3. LOAD MODELS ---
print(f"Loading Pre-Trained CLIP to {DEVICE_YOLO}...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE_YOLO)

if 'blip_model' in globals() and 'blip_processor' in globals():
    print(f"BLIP-2 is already loaded in memory! Skipping download/initialization.")
else:
    print(f"Loading Frozen BLIP-2 to {DEVICE_BLIP}...")
    blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
    blip_model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b", 
        torch_dtype=torch.float16, 
        device_map=BLIP_DEVICE_MAP
    )

# --- 4. LOAD INDICES ---
index_05 = faiss.read_index(INDEX_PATH_B_05)
index_08 = faiss.read_index(INDEX_PATH_B_08)

# --- 5. EVALUATION LOOP ---
print("\nRunning ULTRA-FAST Evaluation for Part B (Processing both Alphas simultaneously)...")

total_metrics = {alpha: {'Recall': {5: 0, 10: 0, 15: 0}, 'NDCG': {5: 0, 10: 0, 15: 0}, 'mAP': {5: 0, 10: 0, 15: 0}} for alpha in ALPHAS}
valid_queries = 0

for idx, row in tqdm(query_df.iterrows(), total=len(query_df)):
    abs_path = os.path.join(BASE_DIR, "img", row['image_name'])
    if not os.path.exists(abs_path): continue
    valid_queries += 1
    
    cropped_query = get_cropped_image(abs_path)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        
        # 1. FAISS Query Embed (GPU 0)
        clip_inputs = clip_processor(images=cropped_query, return_tensors="pt").to(DEVICE_YOLO)
        vision_out = clip_model.vision_model(pixel_values=clip_inputs.pixel_values)
        query_embed = clip_model.visual_projection(vision_out.pooler_output)
        query_embed = query_embed / query_embed.norm(p=2, dim=-1, keepdim=True)
        query_np = query_embed.to(torch.float32).cpu().numpy()
        
        # 2. Search BOTH indices simultaneously
        _, idx_05 = index_05.search(query_np, 15)
        _, idx_08 = index_08.search(query_np, 15)
        
        candidates_05 = [metadata[i] for i in idx_05[0]]
        candidates_08 = [metadata[i] for i in idx_08[0]]
        
        # 3. Combine unique candidates to save BLIP-2 compute time!
        unique_candidates_dict = {c['faiss_id']: c for c in candidates_05 + candidates_08}
        unique_candidates = list(unique_candidates_dict.values())
        
        # 4. BLIP-2 Batched Re-Ranking (GPU 1)
        pixel_values = blip_processor(images=cropped_query, return_tensors="pt").pixel_values.to(DEVICE_BLIP, torch.float16)
        pixel_values = pixel_values.expand(len(unique_candidates), -1, -1, -1)
        
        captions = [c['generated_caption'] for c in unique_candidates]
        text_inputs = blip_processor.tokenizer(captions, return_tensors="pt", padding=True, truncation=True).to(DEVICE_BLIP)
        
        input_ids, attention_mask = text_inputs.input_ids, text_inputs.attention_mask
        labels = input_ids.clone()
        labels[labels == blip_processor.tokenizer.pad_token_id] = -100
        
        outputs = blip_model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_matrix = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='none')
        loss_matrix = loss_matrix.view(shift_labels.size())
        mask = (shift_labels != -100)
        losses = ((loss_matrix * mask).sum(dim=1) / mask.sum(dim=1)).tolist()
        
        # Create a fast lookup dictionary for the scores
        score_lookup = {unique_candidates[i]['faiss_id']: losses[i] for i in range(len(unique_candidates))}
        
        # 5. Split, sort, and calculate metrics for Alpha 0.5
        reranked_05 = [(score_lookup[c['faiss_id']], c) for c in candidates_05]
        reranked_05.sort(key=lambda x: x[0])
        final_05 = [item[1]['item_id'] for item in reranked_05]
        metrics_05 = calculate_metrics(final_05, row['item_id'], k_values)
        for m in ['Recall', 'NDCG', 'mAP']:
            for k in k_values: total_metrics[0.5][m][k] += metrics_05[m][k]
                
        # 6. Split, sort, and calculate metrics for Alpha 0.8
        reranked_08 = [(score_lookup[c['faiss_id']], c) for c in candidates_08]
        reranked_08.sort(key=lambda x: x[0])
        final_08 = [item[1]['item_id'] for item in reranked_08]
        metrics_08 = calculate_metrics(final_08, row['item_id'], k_values)
        for m in ['Recall', 'NDCG', 'mAP']:
            for k in k_values: total_metrics[0.8][m][k] += metrics_08[m][k]

# --- 6. OUTPUT FINAL RESULTS FOR BOTH ALPHAS ---
for alpha in ALPHAS:
    print(f"\nCONDITION B: Frozen CLIP + BLIP-2 (Alpha = {alpha})")
    results_df = pd.DataFrame(total_metrics[alpha]) / valid_queries
    results_df.index = [f"@{k}" for k in k_values]
    display(HTML(results_df.to_html(classes='table table-striped table-bordered text-center', float_format="%.4f")))

Checking GPU availability...
Loading Pre-Trained CLIP to cuda:0...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

  0%|          | 0/14218 [00:00<?, ?it/s]


CONDITION B: Frozen CLIP + BLIP-2 (Alpha = 0.5)


,Recall,NDCG,mAP
@5,0.2137,0.1501,0.1007
@10,0.3624,0.2318,0.1166
@15,0.4769,0.2997,0.1213



CONDITION B: Frozen CLIP + BLIP-2 (Alpha = 0.8)


,Recall,NDCG,mAP
@5,0.2326,0.1676,0.1100
@10,0.4000,0.2641,0.1278
@15,0.5308,0.3487,0.1332


## Part C - Done in VR_Final_C.ipynb